# Claude Agent SDK: sessions (continue, resume, fork)

A **session** is the conversation history the SDK accumulates while your agent works: your prompt, every tool call, every tool result, every reply. The SDK writes it to disk automatically so you can **come back later with full context**: files already read, analysis already done, decisions already made.

**What you will learn (and see running):**
1. Where sessions live on disk, and what is inside one
2. Capturing a `session_id`
3. **Resume**: return to a specific session
4. **Continue**: return to the most recent session
5. **`ClaudeSDKClient`**: automatic multi-turn in one process
6. **Fork**: branch a session without touching the original
7. Recovering from `max_turns` / budget limits
8. Listing and reading sessions, and what to do across machines

> Sessions persist the **conversation, not the filesystem**. If the agent edited files, those edits stay on disk regardless of which session you resume. (To undo file edits, use *file checkpointing*.)

## Which approach do I need?

| What you are building | Use |
|---|---|
| One-shot task, no follow-up | Nothing extra: one `query()` call |
| Multi-turn chat in **one process** | `ClaudeSDKClient` (Python tracks the session for you) |
| Pick up after a **process restart**, most recent conversation | `continue_conversation=True` |
| Return to a **specific** past session (e.g. one per user) | Capture the ID and pass `resume=` |
| Try an alternative **without losing the original** | `resume=` + `fork_session=True` |
| Stateless, nothing written to disk | Python: set `CLAUDE_CODE_SKIP_PROMPT_HISTORY` in `env` (TypeScript has `persistSession: false`) |

**Continue vs resume:** both pick up an existing session and add to it. *Continue* finds the most recent one in the current directory (you track nothing). *Resume* takes an explicit ID (you track the ID).

**Fork** is different: it creates a **new** session that starts with a **copy** of the original's history. The original stays unchanged.

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("sessions-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

In [ ]:
import re, os, json
from pathlib import Path
from collections import Counter

(DEMO_DIR / "notes.txt").write_text("The launch code name is PELICAN-7.\n")
(DEMO_DIR / "todo.txt").write_text("1. buy milk\n2. ship the release\n")
print("Created:", sorted(p.name for p in DEMO_DIR.iterdir()))

In [ ]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, ClaudeSDKClient,
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
)

def short(value, n=110):
    text = str(value).replace("\n", " ")
    return text if len(text) <= n else text[: n - 1] + "…"


async def ask(prompt, **opts):
    """Run one query in the demo folder. Returns (answer_text, session_id)."""
    opts.setdefault("cwd", str(DEMO_DIR))
    opts.setdefault("allowed_tools", ["Read", "Glob"])
    answer, session_id, tools = "", None, []
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, AssistantMessage):
                tools += [b.name for b in m.content if isinstance(b, ToolUseBlock)]
            elif isinstance(m, ResultMessage):
                session_id = m.session_id          # present on EVERY result, even errors
                if m.subtype == "success":
                    answer = m.result
                else:
                    print("   result subtype:", m.subtype)
    except Exception as error:
        print("   (query raised:", short(error, 120), ")")
    print("   tools used :", tools or "none")
    print("   answer     :", short(answer, 300))
    print("   session_id :", session_id)
    return answer, session_id

## 1. Where sessions live on disk

Claude Code stores each session as a **JSONL file** (one JSON object per line):

```
~/.claude/projects/<encoded-cwd>/<session-id>.jsonl
```

`<encoded-cwd>` is the absolute working directory with **every non-alphanumeric character replaced by `-`** (`/Users/me/proj` becomes `-Users-me-proj`). If you set `CLAUDE_CONFIG_DIR`, look under `$CLAUDE_CONFIG_DIR/projects/` instead. (Very long paths are truncated with a hash.)

Because the folder is derived from `cwd`, **the same demo folder always maps to the same place**. Let's compute it and write a helper that lists the transcripts.

In [ ]:
def transcript_dir(cwd=DEMO_DIR):
    base = Path(os.environ.get("CLAUDE_CONFIG_DIR", Path.home() / ".claude"))
    return base / "projects" / re.sub(r"[^A-Za-z0-9]", "-", str(cwd))

def show_transcripts():
    d = transcript_dir()
    print("transcript folder:", d)
    files = sorted(d.glob("*.jsonl"), key=lambda p: p.stat().st_mtime) if d.exists() else []
    if not files:
        print("  (no sessions yet)")
    for p in files:
        print(f"  {p.name}  {p.stat().st_size:>7} bytes  {sum(1 for _ in p.open())} lines")
    return files

show_transcripts()

## 2. Capture the session ID

The ID is on the `session_id` field of the **`ResultMessage`**, which is present on every result, successful or not. (In TypeScript it is also on the init `SystemMessage`.)

We ask the agent to read a file so the session contains a tool call, a tool result and an answer.

In [ ]:
answer1, session_id = await ask("Read notes.txt and tell me the launch code name in one short sentence.")

In [ ]:
files = show_transcripts()

# Peek inside the newest transcript: what kinds of entries does a session hold?
newest = files[-1]
kinds = Counter()
for line in newest.open():
    kinds[json.loads(line).get("type", "?")] += 1
print("\nentry types in", newest.name)
print(dict(kinds))

## 3. Control experiment: a brand-new query has no memory

A new `query()` without `resume` starts a **fresh** session. It cannot know what happened before. (This also creates a second transcript file.)

In [ ]:
answer_fresh, fresh_id = await ask(
    "What is the launch code name? If you do not already know it, answer exactly 'unknown'. "
    "Do not use any tools."
)
print("\nsame session as before?", fresh_id == session_id)

## 4. Resume by ID

`resume=<session_id>` restores **everything**: the file it read, its reasoning, its answer. Typical reasons to resume:

- **Follow up** on a finished task without re-reading files
- **Recover from a limit** (`error_max_turns` / `error_max_budget_usd`)
- **Restart your process** after saving the ID

In [ ]:
answer_resumed, resumed_id = await ask(
    "What was the launch code name? Answer from memory only. Do not use any tools.",
    resume=session_id,
)
print("\nsame session id after resume?", resumed_id == session_id)

The agent answers correctly **without calling a tool**, because the earlier tool result is part of the restored history. The same prompt in the control experiment failed.

> Note: resuming keeps the **same** session ID and appends to the same transcript. (Fork, below, creates a new one.)

You can resume from **any working directory** on the same machine: Claude Code searches beyond the current project folder to find the ID. The session file still has to exist on this machine.

## 5. Continue the most recent session

`continue_conversation=True` needs no ID: it finds the **most recent session in the current directory**. Handy for "pick up where we left off" after a restart, when your app runs one conversation at a time.

In [ ]:
answer_cont, cont_id = await ask(
    "Add the number 1 to the digit in that code name. Reply with just the new code name. No tools.",
    continue_conversation=True,
)
print("\ncontinued the same session?", cont_id == session_id)

## 6. Automatic multi-turn with `ClaudeSDKClient`

If you are chatting in **one process**, `ClaudeSDKClient` handles the session for you: each `client.query()` continues the same session, no IDs involved. Use it as an async context manager so connection setup and teardown are automatic.

(In TypeScript the equivalent is passing `continue: true` on each follow-up `query()`.)

In [ ]:
def show(message):
    """Print only the human-readable parts of a message."""
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print("  Claude:", short(block.text, 300))
            elif isinstance(block, ToolUseBlock):
                print("  [tool]", block.name)
    elif isinstance(message, ResultMessage):
        cost = f"${message.total_cost_usd:.4f}" if message.total_cost_usd is not None else "N/A"
        print(f"  [done: {message.subtype}, cost: {cost}]")


options = ClaudeAgentOptions(cwd=str(DEMO_DIR), allowed_tools=["Read", "Glob"])

async with ClaudeSDKClient(options=options) as client:
    print("Turn 1")
    await client.query("Read todo.txt and count the items. Answer with just the number.")
    async for message in client.receive_response():
        show(message)

    print("Turn 2 (same session, no ID passed)")
    await client.query("What was the second item on that list? No tools.")
    async for message in client.receive_response():
        show(message)

## 7. Fork: explore an alternative without losing the original

Forking creates a **new session with a copy of the original's history**. From there the two diverge independently. The fork gets its own ID; the original's ID and history are untouched.

Experiment, using the session from section 2 (which has read `notes.txt` but has **never been told a project name**):

1. **Fork** it and tell the fork: "the project is called Beta".
2. **Resume the original** and ask for the project name → it should say it was never told.
3. **Resume the fork** and ask → it should say *Beta*.

In [ ]:
print("--- step 1: fork and name the project 'Beta' ---")
_, fork_id = await ask(
    "From now on, the project is called Beta. Reply with just: OK.",
    resume=session_id,
    fork_session=True,
)
print("\nfork id differs from original?", fork_id != session_id)

In [ ]:
QUESTION = ("What name did I tell you the project has? Answer in one word, or 'never told' "
            "if I did not say. Do not use tools.")

print("--- step 2: original session ---")
await ask(QUESTION, resume=session_id)

print("\n--- step 3: forked session ---")
await ask(QUESTION, resume=fork_id)

In [ ]:
show_transcripts();

Two independent histories, two transcript files, from one starting point.

> **Forking branches the conversation, not the filesystem.** If a forked agent edits files, the change is real and visible to every session in that folder.

## 8. Recovering from a limit

When a run stops at `max_turns` or `max_budget_usd`, the `ResultMessage` still carries the `session_id`. **Resume with a higher limit** and the agent continues with all its prior work. (A single-shot `query()` *raises* after yielding the error result, so catch it first. Our `ask` helper already does.)

In [ ]:
print("--- first attempt: only 1 tool turn allowed ---")
_, limited_id = await ask(
    "Read notes.txt, then read todo.txt, then tell me the code name and the number of todo items.",
    max_turns=1,
)

In [ ]:
print("--- resume with a higher limit ---")
await ask(
    "Please finish the task you were working on.",
    resume=limited_id,
    max_turns=10,
)

## 9. Inspecting sessions from code

Both SDKs can **enumerate sessions and read their messages**: useful for session pickers, cleanup jobs, or transcript viewers.

| Function | Purpose |
|---|---|
| `list_sessions(directory=...)` | Sessions on disk for a project folder |
| `get_session_messages(session_id, directory=...)` | The message chain the agent would see on resume |
| `get_session_info(session_id)` | Metadata for one session |
| `rename_session(session_id, title)` | Give a session a readable title |
| `tag_session(session_id, tag)` | Tag it for organizing |

In [ ]:
from claude_agent_sdk import list_sessions, get_session_messages, get_session_info, rename_session, tag_session

sessions = list_sessions(directory=str(DEMO_DIR))
print(f"{len(sessions)} sessions found\n")
for s in sessions:
    print(s.session_id[:8], "|", short(s.summary, 60), "| first prompt:", short(s.first_prompt, 50))

In [ ]:
# Give the original session a human-friendly title and a tag, then read it back.
rename_session(session_id, "Launch notes (original)", directory=str(DEMO_DIR))
tag_session(session_id, "demo", directory=str(DEMO_DIR))
info = get_session_info(session_id, directory=str(DEMO_DIR))
print("title:", info.custom_title, "| tag:", info.tag)

print("\nmessages in the original session:")
for msg in get_session_messages(session_id, directory=str(DEMO_DIR)):
    print(f"  {msg.type:9}", short(msg.message, 100))

Notice `get_session_messages` returns the **linked message chain the agent would see**. After auto-compaction, earlier turns are replaced by a summary, so the count can be much smaller than the raw transcript.

## 10. Sessions across machines

Session files are **local to the machine** that created them. To resume elsewhere (CI workers, containers, serverless), pick one:

1. **Pass a session store.** Attach a `session_store` adapter so the SDK mirrors transcripts to **your own backend** (object store, Redis, Postgres...) and another host can resume them. This is the subject of the next notebook. The store key derives from the working directory, so resume from a matching `cwd`.
2. **Move the file.** Persist `~/.claude/projects/<encoded-cwd>/<session-id>.jsonl` and restore it on the new host before calling `resume`.
3. **Don't rely on resume.** Save what you need (analysis, decisions, diffs) as **application state** and pass it into a fresh session's prompt. Often the most robust option.

## 11. Recap

| Need | Tool |
|---|---|
| Remember a session | `ResultMessage.session_id` |
| Return to a specific one | `resume=<id>` |
| Return to the newest | `continue_conversation=True` |
| Chat in one process | `ClaudeSDKClient` |
| Branch without losing the original | `resume=<id>` + `fork_session=True` |
| Survive a limit | Catch the error, resume with a higher limit |
| Move between machines | `session_store`, or move the `.jsonl`, or pass state in a new prompt |

Sessions store the **conversation**. Files the agent changed are separate.

### Try it yourself
1. Resume the *original* session once more and ask what you did earlier in the conversation. Does it mention the fork? (It should not.)
2. Build a tiny CLI loop that saves `session_id` to a file, so you can quit and `resume` tomorrow.
3. Fork twice from the same session and compare two alternative answers to the same question.
4. Next: **`agent-sdk-session-storage-explained.ipynb`**.